# neuros-astro Pipeline Walkthrough

This notebook demonstrates the complete neuros-astro pipeline:
1. Generate or load calcium imaging data
2. Detect astrocyte calcium events
3. Build functional connectivity networks
4. Tokenize for foundation models
5. Visualize results

**Compute Requirements**: CPU only! Runs easily on any laptop.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# neuros-astro modules
from neuros_astro.io.synthetic import generate_synthetic_astro_traces
from neuros_astro.events.event_detection import detect_events_from_traces
from neuros_astro.networks.functional_connectivity import build_event_coactivation_graph
from neuros_astro.networks.graph_features import compute_graph_summary_features
from neuros_astro.tokenization.event_tokenizer import AstroEventTokenizer
from neuros_astro.tokenization.astro_tokenizer import BinnedAstroTokenizer
from neuros_astro.visualization.event_plots import (
    plot_event_raster,
    plot_event_distributions,
    plot_event_statistics_summary,
    plot_network_graph,
    plot_trace_with_events,
)

print("✓ Imports successful!")

## Step 1: Generate Synthetic Data

We'll start with synthetic data to understand the pipeline.
This simulates slow astrocyte calcium transients.

In [ ]:
# Configuration
n_regions = 20  # Number of astrocyte regions
duration_s = 180.0  # 3 minutes
frame_rate_hz = 10.0  # 10 Hz sampling
session_id = "demo_walkthrough"

# Generate synthetic traces
print("🔬 Generating synthetic astrocyte calcium traces...")
traces, ground_truth_events = generate_synthetic_astro_traces(
    n_regions=n_regions,
    duration_s=duration_s,
    frame_rate_hz=frame_rate_hz,
    n_events_per_region=10,
    seed=42,
)

print(f"✓ Generated traces: shape {traces.shape} [regions, timepoints]")
print(f"✓ Ground truth: {len(ground_truth_events)} events")
print(f"✓ Duration: {duration_s}s at {frame_rate_hz} Hz")

In [ ]:
# Visualize a few example traces
fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
time = np.arange(traces.shape[1]) / frame_rate_hz

for i in range(5):
    axes[i].plot(time, traces[i], 'k-', linewidth=0.8, alpha=0.7)
    axes[i].set_ylabel(f'Region {i}\nΔF/F', fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

axes[-1].set_xlabel('Time (s)', fontsize=12)
axes[0].set_title('Example Synthetic Astrocyte Calcium Traces', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Note the slow calcium transients characteristic of astrocytes")

## Step 2: Detect Calcium Events

Use robust z-score threshold crossing to detect slow calcium events.

In [ ]:
# Detect events
print("🔍 Detecting calcium events...")
events = detect_events_from_traces(
    traces=traces,
    frame_rate_hz=frame_rate_hz,
    session_id=session_id,
    z_threshold=2.0,  # Robust z-score threshold
    min_duration_s=0.8,  # Minimum event duration
    merge_gap_s=0.5,  # Merge events closer than 0.5s
)

print(f"✓ Detected {len(events)} events")
print(f"✓ Detection rate: {len(events)/len(ground_truth_events)*100:.1f}% of ground truth")

# Print statistics
plot_event_statistics_summary(events)

In [ ]:
# Visualize event raster
fig, ax = plot_event_raster(
    events,
    frame_rate_hz=frame_rate_hz,
    figsize=(14, 8),
    show_confidence=True
)
plt.show()

print("\n✓ Blue lines show event duration")
print("✓ Red dots mark peak amplitude")
print("✓ Color indicates detection confidence")

In [ ]:
# Visualize event feature distributions
fig = plot_event_distributions(events, figsize=(14, 10))
plt.show()

print("\n✓ Duration: Astrocyte events are typically 1-10 seconds")
print("✓ Amplitude: ΔF/F shows event strength")
print("✓ Scatter plot shows duration vs amplitude relationship")

In [ ]:
# Zoom in on one region to see detection quality
region_id = events[0].region_id
region_idx = int(region_id.split('_')[1])  # Extract region index

fig, ax = plot_trace_with_events(
    trace=traces[region_idx],
    events=events,
    region_id=region_id,
    frame_rate_hz=frame_rate_hz,
    figsize=(14, 5)
)
plt.show()

print(f"\n✓ Showing region {region_id}")
print("✓ Red shading marks detected events")
print("✓ Red dots mark peak times")

## Step 3: Build Functional Connectivity Networks

Construct time-windowed graphs based on event coactivation.

In [ ]:
# Build coactivation networks
print("🕸️  Building astrocyte coactivation networks...")
graphs = build_event_coactivation_graph(
    events=events,
    session_id=session_id,
    frame_rate_hz=frame_rate_hz,
    window_size_s=30.0,  # 30-second windows
    stride_s=10.0,  # 10-second stride (overlapping windows)
    min_edge_weight=0.15,  # Only strong connections
)

print(f"✓ Generated {len(graphs)} network windows")

if len(graphs) > 0:
    # Analyze first graph
    features = compute_graph_summary_features(graphs[0])
    print(f"\nExample graph (first window):")
    print(f"  Nodes: {features['n_nodes']}")
    print(f"  Edges: {features['n_edges']}")
    print(f"  Density: {features['density']:.3f}")
    print(f"  Mean degree: {features['degree_mean']:.2f}")
    print(f"  Connected components: {features['n_connected_components']}")

In [ ]:
# Visualize first network
if len(graphs) > 0:
    fig, ax = plot_network_graph(
        graphs[0],
        figsize=(10, 10),
        layout="spring",
        show_weights=True
    )
    plt.show()
    
    print("\n✓ Nodes represent astrocyte regions")
    print("✓ Edge width shows coactivation strength")
    print("✓ Layout based on connectivity structure")

In [ ]:
# Show network evolution over time
from neuros_astro.visualization.event_plots import plot_network_evolution

fig = plot_network_evolution(graphs, max_graphs=5, figsize=(18, 4))
plt.show()

print("\n✓ Networks evolve over time as different regions coactivate")
print("✓ This captures slow-timescale astrocyte dynamics")

## Step 4: Tokenize for Foundation Models

Convert events and networks into model-ready token sequences.

In [ ]:
# Tokenize events (irregular sampling)
print("🎯 Tokenizing events for foundation models...")
event_tokenizer = AstroEventTokenizer(normalize=True)
event_tokens = event_tokenizer.tokenize(events, session_id=session_id)

print(f"✓ Event tokens: {len(event_tokens.tokens)} events")
print(f"✓ Features per event: {len(event_tokens.feature_names)}")
print(f"\nFeature names:")
for i, name in enumerate(event_tokens.feature_names, 1):
    print(f"  {i}. {name}")

In [ ]:
# Visualize token embeddings (first 3 features)
import numpy as np

if len(event_tokens.tokens) > 0:
    tokens_array = np.array(event_tokens.tokens)
    
    fig = plt.figure(figsize=(12, 4))
    
    for i in range(min(3, tokens_array.shape[1])):
        ax = fig.add_subplot(1, 3, i+1)
        ax.hist(tokens_array[:, i], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
        ax.set_xlabel(event_tokens.feature_names[i], fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'Feature: {event_tokens.feature_names[i]}', fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Token features are normalized for model ingestion")

In [ ]:
# Tokenize with regular binning (alternative representation)
print("\n📊 Creating binned tokens (regular sampling)...")
binned_tokenizer = BinnedAstroTokenizer(bin_size_s=5.0, normalize=True)
binned_tokens = binned_tokenizer.tokenize(
    events=events,
    duration_s=duration_s,
    session_id=session_id,
    graphs=graphs
)

print(f"✓ Binned tokens: {len(binned_tokens.tokens)} time bins")
print(f"✓ Bin size: 5.0 seconds")
print(f"✓ Features per bin: {len(binned_tokens.feature_names)}")
print(f"\nBinned features:")
for i, name in enumerate(binned_tokens.feature_names, 1):
    print(f"  {i}. {name}")

## Step 5: Export for neuroFMx Integration

Save tokens in formats ready for foundation model training.

In [ ]:
from neuros_astro.export.to_neurofm import (
    save_tokenized_sequence_npz,
    build_neurofm_manifest,
    save_neurofm_manifest,
)
from neuros_astro.export.to_parquet import save_events_parquet

# Create output directory
output_dir = Path("./notebook_outputs")
output_dir.mkdir(exist_ok=True)

# Save events
events_path = output_dir / "events.parquet"
save_events_parquet(events, events_path)
print(f"✓ Events saved: {events_path}")

# Save event tokens
event_tokens_path = output_dir / "event_tokens.npz"
save_tokenized_sequence_npz(event_tokens, event_tokens_path)
print(f"✓ Event tokens saved: {event_tokens_path}")

# Save binned tokens
binned_tokens_path = output_dir / "binned_tokens.npz"
save_tokenized_sequence_npz(binned_tokens, binned_tokens_path)
print(f"✓ Binned tokens saved: {binned_tokens_path}")

# Create neuroFMx manifest
manifest = build_neurofm_manifest(
    session_id=session_id,
    modalities={
        "astro_events": {
            "type": "event_tokens",
            "path": str(event_tokens_path.name),
            "sampling": "irregular",
            "timestamp_key": "timestamps_s",
        },
        "astro_binned": {
            "type": "binned_tokens",
            "path": str(binned_tokens_path.name),
            "sampling": "regular",
            "bin_size_s": 5.0,
        }
    },
    metadata={
        "n_events": len(events),
        "n_regions": n_regions,
        "n_graphs": len(graphs),
        "duration_s": duration_s,
    }
)

manifest_path = output_dir / "neurofm_manifest.json"
save_neurofm_manifest(manifest, manifest_path)
print(f"✓ Manifest saved: {manifest_path}")

print("\n🎉 All outputs ready for neuroFMx integration!")

## Summary

You've completed the full neuros-astro pipeline!

**What we did:**
1. ✅ Generated synthetic astrocyte calcium traces
2. ✅ Detected slow calcium events (1-10s duration)
3. ✅ Built time-windowed coactivation networks
4. ✅ Tokenized events for foundation models
5. ✅ Exported data in neuroFMx-compatible formats

**Next steps:**
- Try with real Allen Brain Observatory data (see notebook 02)
- Integrate with neuroFMx for training (see notebook 03)
- Run ablation experiments (neural vs neural+astro)

**Key insight:**
This entire pipeline runs on CPU and completes in seconds!
No GPU needed for data processing. 🚀